# Assignment 2 — Version C

 **Don't rush — this assignment covers a lot. Read each section carefully, understand what the code is actually doing, and build at your own pace.**


# Module 1: Ingesting Raw Documents

Data ingestion is the entry point of a Retrieval-Augmented Generation (RAG) system. Before anything else can happen, we must read files from storage and parse them into structured document objects. Each object carries two things: the raw page text and metadata (descriptive info like filename and page number). Think of this step as scanning and cataloguing books before putting them on a searchable shelf.


We are focused on PDFs in this module. The full project will extend this to CSV, XLSX, DOCX, and TXT formats.
Try adding support for one extra format on your own — it's great practice!

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

In [ ]:
def ingest_pdfs(folder_path):

    document_collection = []
    pdf_files = Path(folder_path).glob("**/*.pdf")
    for pdf_path in pdf_files:                  
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()  
        for page in pages:
            page.metadata["filename"] = pdf_path.name
            page.metadata["source_path"] = str(pdf_path) 
            if "page" in page.metadata:
                page.metadata["page_number"] = page.metadata["page"]
            document_collection.append(page)
    return document_collection

In [ ]:
raw_documents = ingest_pdfs("pdf")
raw_documents

# Module 2: Splitting Documents into Manageable Chunks

Large documents need to be broken into smaller segments before they can be embedded and searched. This process is called chunking. LLMs can only process a limited amount of text at a time (their context window), and retrieval is more accurate when chunks are focused and concise. **RecursiveCharacterTextSplitter** splits text intelligently — first at paragraph breaks, then line breaks, then spaces — and uses an overlap between consecutive chunks to preserve meaning across boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def create_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    text_chunks = text_splitter.split_documents(documents)
    return text_chunks

In [ ]:
text_chunks = create_chunks(raw_documents)
text_chunks

# Module 3: Generating Embeddings

An embedding is a numerical fingerprint of a piece of text — a list of floating point numbers that represents its meaning in a mathematical space. Text with similar semantics will produce vectors that are geometrically close. We use the `all-MiniLM-L6-v2` model from the `sentence-transformers` library to generate these embeddings locally.

---

**`from sentence_transformers import SentenceTransformer`**

This imports the core model class. What this library does:
- Fetches pre-trained models from HuggingFace
- Encodes any text string into a fixed-size dense vector

Internally, it uses HuggingFace Transformers and PyTorch as its foundation.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"Fetching model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print("Model is ready to use")
        embed_dim = self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model must be loaded before generating embeddings")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

# Module 4: Persisting Embeddings in a Vector Store

A Vector Store is a database built specifically for storing and searching embedding vectors. Rather than matching keywords, it finds the documents whose vectors are closest to your query vector. ChromaDB runs entirely on your machine, requires no cloud subscription, and writes data to disk so it persists between sessions.

In [ ]:
class VectorStore:
    """Persists and retrieves document chunks using ChromaDB as the vector backend"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings"}
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Mismatch: number of documents and embeddings must be identical")
        record_ids = []
        metadatas = []
        doc_texts = []
        embedding_list = []
        for i, doc in enumerate(documents):
            record_ids.append(uuid.uuid4().hex[:8])
            meta = dict(doc.metadata)
            meta["doc_index"] = i
            meta["content_length"] = len(doc.page_content)
            metadatas.append(meta)
            doc_texts.append(doc.page_content)
            embedding_list.append(embeddings[i].tolist())
        self.collection.add(
            ids=record_ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=doc_texts
        )

## Encode chunks and insert into the vector store

In [ ]:
raw_texts = [chunk.page_content for chunk in text_chunks]
raw_texts

In [ ]:
embedding_manager = EmbeddingManager()
chunk_embeddings = embedding_manager.generate_embeddings(raw_texts)
vectorstore = VectorStore()
vectorstore.add_documents(text_chunks, chunk_embeddings)

# Module 5: Encoding the User Query

When a user asks a question, that question must be converted into an embedding using the same model we used for documents. Only then can we meaningfully compare it against stored vectors. The query embedding is passed to the vector store which searches for the most similar document chunks.

---

# Module 6: Cosine Similarity and Ranking

Cosine similarity measures the angle between two vectors — a score of 1.0 means perfect alignment (identical meaning), while 0.0 means completely unrelated. The vector store ranks all stored chunks by this score and returns the top_k most relevant ones. A minimum score threshold can be applied to filter out weak matches.

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        query_vec = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_vec.tolist()],
            n_results=top_k
        )
        matching_docs = []
        total = len(results["ids"][0])
        for i in range(total):
            dist = results["distances"][0][i]
            score = 1 - dist
            if score >= score_threshold:
                matching_docs.append({
                    "id": results["ids"][0][i],
                    "content": results["documents"][0][i],
                    "metadata": results["metadatas"][0][i],
                    "similarity_score": score,
                    "distance": dist,
                    "rank": i + 1
                })
        return matching_docs

In [ ]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever.retrieve("put something here similar to your data to retrieve")

# Module 7: Prompt Construction and LLM Response

With relevant chunks retrieved, the final step is crafting a prompt and sending it to the LLM. The prompt gives the model explicit context (the retrieved passages) alongside the user's question. This grounds the model's response in actual document content, reducing hallucination and improving accuracy.

In [ ]:
def rag_simple(query, retriever, llm, top_k=3):
    top_chunks = retriever.retrieve(query=query, top_k=top_k)
    combined_context = "\n\n".join(chunk["content"] for chunk in top_chunks)
    prompt_text = f"""Answer the question using only the context provided below.
Context: {combined_context}
Question: {query}
Answer:
"""
    llm_response = llm.invoke(prompt_text)
    return llm_response.content

In [ ]:
final_answer = rag_simple("three reasons for forgetting", rag_retriever, llm)  # import your llm first
print(final_answer)

# Module 8: Production-Grade RAG (Optional)

This section upgrades the basic pipeline with features that make it ready for real-world deployment: streaming output so users see partial answers in real time, source citations tied to each claim, a session history log for conversational continuity, and an optional summarizer to compress lengthy responses into key takeaways.

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    top_results = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)

    combined_context = "\n\n".join(doc["content"] for doc in top_results)
    source_list = []
    for doc in top_results:
        source_list.append({
            "source_file": doc["metadata"].get("filename", "Unknown"),
            "page": doc["metadata"].get("page_number", "Unknown"),
            "sim_score": doc["similarity_score"],
            "preview": doc["content"][:150]
        })
    top_confidence = 0
    if top_results:
        top_confidence = max(doc["similarity_score"] for doc in top_results)

    prompt_text = f"""
Answer the question using only the context provided.
Context: {combined_context}
Question: {query}
Answer:
"""
    response = llm.invoke(prompt_text)
    output = {"answer": response.content, "sources": source_list, "confidence": top_confidence}

    if return_context:
        output["context"] = combined_context

    return output

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.session_history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        top_results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        context_block = "\n\n".join(doc["content"] for doc in top_results)
        citation_data = []
        for doc in top_results:
            citation_data.append({
                "source_file": doc["metadata"].get("filename", "Unknown"),
                "page": doc["metadata"].get("page_number", "Unknown"),
                "sim_score": doc["similarity_score"]
            })

        full_prompt = f"""
Answer the question using only the context provided.
Context: {context_block}
Question: {question}
Answer:
"""
        if stream:
            for i in range(0, len(full_prompt), 200):
                print(full_prompt[i:i+200])
                time.sleep(0.05)

        response = self.llm.invoke(full_prompt)
        answer_text = response.content
        citation_lines = []
        for src in citation_data:
            citation_lines.append(f"{src['source_file']} (Page {src['page']})")

        cited_answer = answer_text + "\n\nSources:\n" + "\n".join(citation_lines)

        brief_summary = None
        if summarize:
            summary_request = f"Summarize the following answer in 2 sentences:\n\n{answer_text}"
            brief_summary = self.llm.invoke(summary_request).content

        entry = {"question": question, "answer": cited_answer, "sources": citation_data, "summary": brief_summary}
        self.session_history.append(entry)
        return {
            "question": question,
            "answer": cited_answer,
            "sources": citation_data,
            "summary": brief_summary,
            "history": self.session_history
        }